> **Version corrigée** — ce notebook contient le code complet de tous les exercices, exécuté de bout en bout, ainsi qu'un élément de réponse pour chaque question d'observation. La version étudiant (à compléter soi-même) est téléchargeable depuis la page du cours.

# Bagging et forêts aléatoires

*Notebook 8/9 — "Introduction à l'apprentissage supervisé" (L3 MIASHS → Master, Guillaume Metzler, Université Lyon 2).*

Jusqu'ici, chaque notebook entraînait un modèle unique. On s'intéresse maintenant à des méthodes qui **combinent plusieurs modèles** pour construire un modèle final plus performant. Ce notebook traite de la famille des méthodes qui agrègent des modèles **en parallèle** pour réduire la variance :

- le **bagging** (*bootstrap aggregating*), qui entraîne un même type de modèle sur plusieurs échantillons bootstrap et moyenne les résultats ;
- les **forêts aléatoires**, un bagging d'arbres avec une randomisation supplémentaire sur les variables ;
- l'estimation **out-of-bag (OOB)** et l'**importance des variables**.

Le notebook suivant (9/9) traite du **boosting** (Adaboost, gradient boosting), qui combine des modèles de façon très différente — séquentielle, et orientée réduction du biais plutôt que de la variance.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import make_moons, make_classification, load_wine, load_breast_cancer, load_digits, load_diabetes
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier, RandomForestRegressor
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

RNG = 42
np.random.seed(RNG)
plt.rcParams["figure.dpi"] = 100


## 1. Le principe du bagging

### 1.1 Biais, variance, et modèles instables

On peut décomposer l'écart entre le risque d'une hypothèse $h$ et le risque de Bayes en un terme de **biais** (la meilleure performance atteignable dans la classe $\mathcal H$ considérée) et un terme de **variance** (la sensibilité de l'algorithme aux fluctuations du jeu d'entraînement). Un algorithme à forte variance produit des modèles très différents lorsqu'on le ré-entraîne sur un jeu de données légèrement différent, mais issu de la même distribution.

Les arbres de décision profonds (non élagués) sont un exemple typique : biais faible, mais variance très élevée. C'est précisément ce que le bagging va exploiter.


In [ ]:
# On simule un phénomène de régression : plusieurs jeux d'entraînement de
# même taille, tirés de la même distribution, donnent des modèles très
# différents (à gauche). Leur moyenne (à droite) se rapproche beaucoup plus
# de la fonction vraie.
def fonction_vraie(x):
    return np.sin(2 * np.pi * x) + 0.3 * np.sin(6 * np.pi * x)

rng = np.random.RandomState(RNG)
x_pool = rng.uniform(0, 1, size=300)
y_pool = fonction_vraie(x_pool) + rng.normal(scale=0.25, size=300)
x_grid = np.linspace(0, 1, 300)

degree = 9
n_models = 10
predictions = []

fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharex=True, sharey=True)
for i in range(n_models):
    idx = rng.choice(len(x_pool), size=20, replace=False)
    x_s, y_s = x_pool[idx], y_pool[idx]
    coeffs = np.polyfit(x_s, y_s, degree)
    y_pred = np.polyval(coeffs, x_grid)
    predictions.append(y_pred)
    axes[0].plot(x_grid, y_pred, alpha=0.7)

axes[0].scatter(x_pool, y_pool, s=8, color="gray", alpha=0.3)
axes[0].set_ylim(-3, 3)
axes[0].set_title(f"{n_models} régressions polynomiales (degré {degree})\nsur des échantillons de taille 20, même distribution")
axes[0].set_xlabel("x")
axes[0].set_ylabel("y")

mean_pred = np.mean(predictions, axis=0)
axes[1].plot(x_grid, fonction_vraie(x_grid), "k--", lw=1.5, label="fonction vraie")
axes[1].plot(x_grid, mean_pred, color="tab:red", lw=2, label="moyenne des 10 modèles")
axes[1].scatter(x_pool, y_pool, s=8, color="gray", alpha=0.3)
axes[1].set_ylim(-3, 3)
axes[1].set_xlabel("x")
axes[1].set_title("Modèle moyenné")
axes[1].legend()
plt.tight_layout()
plt.show()


$$ $$

**Question 1 :** En comparant les deux panneaux, que dire de la dispersion des courbes individuelles par rapport à la courbe moyenne ? Cette dernière se rapproche-t-elle de la fonction vraie (biais) ou reste-t-elle éloignée ?

$$ $$

*Éléments de réponse.* Les 10 modèles individuels (gauche) sont très dispersés autour de la fonction vraie : c'est la variance, causée par le petit nombre d'exemples (20) et le degré élevé du polynôme. Une fois moyennés (droite), les erreurs individuelles se compensent en partie et la courbe moyenne colle beaucoup mieux à la fonction vraie, sans que l'on ait modifié la classe de modèles utilisée : on a réduit la variance sans changer le biais.

### 1.2 Le bootstrap

En pratique on ne dispose que d'**un seul** jeu d'entraînement $S$ de taille $m$, pas de plusieurs jeux issus de la même distribution. Le **bootstrap** permet de simuler cette diversité : on tire, avec **remise**, $m$ exemples de $S$ (chaque exemple ayant la même probabilité $1/m$ à chaque tirage). Un même exemple peut donc apparaître plusieurs fois dans un échantillon bootstrap, et d'autres n'y figurer jamais.

**Algorithme (bagging)**

- Pour $t = 1, \ldots, T$ :
  1. tirer un échantillon bootstrap $S_t$ de taille $m$ à partir de $S$ ;
  2. entraîner une hypothèse $h_t$ sur $S_t$ ;
- renvoyer $H_T(x) = \dfrac{1}{T}\sum_{t=1}^T h_t(x)$ (moyenne en régression, vote majoritaire en classification).

Un résultat classique (inégalité de Jensen) montre que l'erreur quadratique moyenne de $H_T$ est toujours inférieure ou égale à l'erreur moyenne des $h_t$ pris séparément, et que le gain est d'autant plus important que les $h_t$ ont une **variance élevée** — ce qui justifie d'utiliser des arbres profonds comme modèles de base plutôt que des arbres peu profonds ou des modèles linéaires.


In [ ]:
# On tire 6 échantillons bootstrap d'un même jeu de données, et on compare
# les frontières de décision de 6 arbres de décision uniques (à gauche) à
# celles de 6 modèles de bagging (200 arbres agrégés chacun, à droite).
X, y = make_moons(n_samples=250, noise=0.3, random_state=7)

xx, yy = np.meshgrid(np.linspace(X[:, 0].min() - 0.5, X[:, 0].max() + 0.5, 300),
                      np.linspace(X[:, 1].min() - 0.5, X[:, 1].max() + 0.5, 300))
grid = np.c_[xx.ravel(), yy.ravel()]

n_repeats = 6
rng = np.random.RandomState(RNG)

fig, axes = plt.subplots(1, 2, figsize=(12, 5.5), sharex=True, sharey=True)
for ax, title in zip(axes, ["Arbres uniques\n(un par échantillon bootstrap)",
                             "Bagging de 200 arbres\n(un modèle par échantillon bootstrap)"]):
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap="coolwarm", edgecolor="k", s=20, zorder=3)
    ax.set_title(title)
    ax.set_xlabel("$x_1$")
    ax.set_ylabel("$x_2$")

for i in range(n_repeats):
    idx = rng.randint(0, len(X), size=len(X))
    X_boot, y_boot = X[idx], y[idx]

    tree = DecisionTreeClassifier(random_state=i)
    tree.fit(X_boot, y_boot)
    Z = tree.predict(grid).reshape(xx.shape)
    axes[0].contour(xx, yy, Z, levels=[0.5], colors=[plt.cm.tab10(i % 10)], linewidths=1.2)

    bag = BaggingClassifier(DecisionTreeClassifier(), n_estimators=200, random_state=i)
    bag.fit(X_boot, y_boot)
    Z = bag.predict(grid).reshape(xx.shape)
    axes[1].contour(xx, yy, Z, levels=[0.5], colors=[plt.cm.tab10(i % 10)], linewidths=1.2)

plt.tight_layout()
plt.show()


$$ $$

**Question 2 :** Que remarquez-vous en comparant la dispersion des frontières entre les deux panneaux ? Est-ce cohérent avec la démonstration précédente en régression ?

$$ $$

*Éléments de réponse.* Les 6 frontières des arbres uniques (gauche) sont très différentes les unes des autres selon l'échantillon bootstrap utilisé, alors que les 6 frontières obtenues par bagging (droite) sont beaucoup plus stables et se superposent presque. On retrouve exactement le phénomène de réduction de variance observé sur l'exemple de régression : agréger des modèles instables (les arbres) rend le modèle final beaucoup moins sensible au jeu d'entraînement précis sur lequel il a été appris.

In [ ]:
# Sur un jeu de données fixe (un seul split train/test), on compare
# directement l'accuracy d'un arbre unique à celle du bagging correspondant.
X, y = make_classification(n_samples=400, n_features=15, n_informative=8, n_redundant=3,
                            n_clusters_per_class=2, flip_y=0.08, random_state=RNG)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=RNG, stratify=y)

tree = DecisionTreeClassifier(random_state=RNG)
tree.fit(X_train, y_train)

bag = BaggingClassifier(DecisionTreeClassifier(), n_estimators=100, random_state=RNG)
bag.fit(X_train, y_train)

print(f"Accuracy (train) - arbre unique : {accuracy_score(y_train, tree.predict(X_train)):.3f}")
print(f"Accuracy (test)  - arbre unique : {accuracy_score(y_test, tree.predict(X_test)):.3f}")
print(f"Accuracy (train) - bagging      : {accuracy_score(y_train, bag.predict(X_train)):.3f}")
print(f"Accuracy (test)  - bagging      : {accuracy_score(y_test, bag.predict(X_test)):.3f}")


### Exercice 1 — Arbre unique vs bagging

Faites la même chose sur `load_wine` : séparez les données en train (70 %) et test (30 %) avec `random_state=42` et `stratify=y`, entraînez un `DecisionTreeClassifier` non contraint et un `BaggingClassifier(DecisionTreeClassifier(), n_estimators=100, random_state=42)`, puis comparez leur accuracy sur le test.


In [ ]:
X_wine, y_wine = load_wine(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X_wine, y_wine, test_size=0.3, random_state=42, stratify=y_wine)

tree = DecisionTreeClassifier(random_state=42)
tree.fit(X_train, y_train)

bag = BaggingClassifier(DecisionTreeClassifier(), n_estimators=100, random_state=42)
bag.fit(X_train, y_train)

acc_tree = accuracy_score(y_test, tree.predict(X_test))
acc_bag = accuracy_score(y_test, bag.predict(X_test))

print(f"Accuracy arbre unique : {acc_tree:.3f}")
print(f"Accuracy bagging      : {acc_bag:.3f}")

assert 0.0 <= acc_tree <= 1.0 and 0.0 <= acc_bag <= 1.0


### 1.3 Validation out-of-bag (OOB)

Le tirage avec remise implique que, pour un échantillon bootstrap donné, une fraction des exemples d'origine ne sont **jamais tirés** : la probabilité qu'un exemple précis ne soit jamais tiré parmi $m$ tirages avec remise dans un ensemble de taille $m$ vaut $\left(1 - \frac1m\right)^m$, qui tend vers $e^{-1} \approx 0.37$ quand $m$ grandit. Ces exemples, dits **out-of-bag** (OOB), n'ont pas servi à entraîner l'hypothèse $h_t$ correspondante : on peut donc les utiliser pour estimer sa capacité de généralisation, un peu comme un jeu de validation obtenu "gratuitement".


In [ ]:
m = 200
proba_non_tire = (1 - 1 / m) ** m
print(f"m = {m}  ->  (1 - 1/m)^m = {proba_non_tire:.4f}  (à comparer à e^-1 = {np.exp(-1):.4f})")


### Exercice 2 — Convergence vers $e^{-1}$

Faites varier $m$ dans `[5, 10, 50, 100, 500, 1000, 5000]`, calculez $(1-1/m)^m$ pour chaque valeur, et tracez la courbe obtenue en fonction de $m$ (axe des $m$ en échelle logarithmique) avec une ligne horizontale à $e^{-1}$ pour comparaison.


In [ ]:
m_values = [5, 10, 50, 100, 500, 1000, 5000]
probas = [(1 - 1 / m) ** m for m in m_values]

plt.figure(figsize=(7, 5))
plt.plot(m_values, probas, marker="o", label=r"$(1-1/m)^m$")
plt.axhline(np.exp(-1), color="gray", linestyle="--", label=r"$e^{-1}$")
plt.xscale("log")
plt.xlabel("m (échelle log)")
plt.ylabel("Probabilité qu'un exemple ne soit jamais tiré")
plt.title("Convergence de $(1-1/m)^m$ vers $e^{-1}$")
plt.legend()
plt.tight_layout()
plt.show()

np.testing.assert_allclose(probas[-1], np.exp(-1), atol=0.01)


In [ ]:
# scikit-learn calcule directement le score OOB via oob_score=True.
X, y = make_classification(n_samples=400, n_features=15, n_informative=8, n_redundant=3,
                            n_clusters_per_class=2, flip_y=0.08, random_state=RNG)

forest_oob = RandomForestClassifier(n_estimators=200, oob_score=True, random_state=RNG)
forest_oob.fit(X, y)
print(f"Score OOB : {forest_oob.oob_score_:.3f}")


### Exercice 3 — Score OOB vs validation croisée

Sur `load_wine`, entraînez un `RandomForestClassifier(n_estimators=300, oob_score=True, random_state=42)` sur l'**ensemble** des données. Affichez son score OOB (`oob_score_`), puis comparez-le à l'accuracy moyenne obtenue par validation croisée à 5 plis (`cross_val_score`) d'un `RandomForestClassifier(n_estimators=300, random_state=42)` équivalent (sans `oob_score`).


In [ ]:
X_wine, y_wine = load_wine(return_X_y=True)

forest_oob = RandomForestClassifier(n_estimators=300, oob_score=True, random_state=42)
forest_oob.fit(X_wine, y_wine)

cv_scores = cross_val_score(
    RandomForestClassifier(n_estimators=300, random_state=42), X_wine, y_wine, cv=5)

print(f"Score OOB                       : {forest_oob.oob_score_:.3f}")
print(f"Accuracy moyenne (CV à 5 plis)  : {cv_scores.mean():.3f} (+/- {cv_scores.std():.3f})")


## 2. Forêts aléatoires

Les **forêts aléatoires** (*Random Forests*, [Breiman, 2001]) sont un cas particulier de bagging appliqué à des arbres de décision, avec une randomisation **supplémentaire** : à chaque nœud, seul un sous-ensemble aléatoire de `max_features` variables (parmi les $d$ disponibles) est considéré pour choisir la meilleure coupure. On parle de **double échantillonnage** : sur les exemples (bootstrap) *et* sur les variables (à chaque nœud).

**Algorithme (forêt aléatoire)**

- Pour $t = 1, \ldots, T$ :
  1. tirer un échantillon bootstrap $S_t$ de taille $m' \le m$ à partir de $S$ ;
  2. construire un arbre $h_t$ où, à chaque nœud, seules `max_features` variables tirées au hasard sont considérées pour la coupure ;
- renvoyer $H_T(x) = \dfrac{1}{T}\sum_{t=1}^T h_t(x)$.

Cette double randomisation crée de la **diversité** entre les arbres (chacun se spécialise sur des régions ou des variables différentes), ce qui renforce la réduction de variance, en plus d'accélérer l'apprentissage de chaque arbre (moins de variables à examiner à chaque split). Les deux principaux hyperparamètres sont donc le nombre d'arbres `n_estimators` et le nombre de variables `max_features`.


In [ ]:
# Évolution de l'erreur OOB en fonction du nombre d'arbres.
X, y = make_classification(n_samples=500, n_features=20, n_informative=8, n_redundant=4,
                            n_clusters_per_class=2, flip_y=0.05, random_state=RNG)

n_estimators_list = [15, 30, 50, 100, 200, 400]
oob_errors = []
for n in n_estimators_list:
    forest = RandomForestClassifier(n_estimators=n, oob_score=True, random_state=RNG)
    forest.fit(X, y)
    oob_errors.append(1 - forest.oob_score_)

plt.figure(figsize=(7, 5))
plt.plot(n_estimators_list, oob_errors, marker="o")
plt.xlabel("Nombre d'arbres (n_estimators)")
plt.ylabel("Erreur OOB (1 - oob_score_)")
plt.title("Évolution de l'erreur OOB avec le nombre d'arbres de la forêt")
plt.tight_layout()
plt.show()


$$ $$

**Question 3 :** Comment évolue l'erreur OOB lorsque n_estimators augmente ? Contrairement à la profondeur d'un arbre, augmenter n_estimators fait-il courir un risque de sur-apprentissage ?

$$ $$

*Éléments de réponse.* L'erreur OOB décroît rapidement avec les premiers arbres ajoutés, puis se stabilise (voire stagne complètement) à mesure que n_estimators continue d'augmenter : ajouter des arbres ne fait qu'améliorer l'estimation de la moyenne $H_T$, sans changer la nature du modèle ni sa complexité individuelle. Contrairement à la profondeur d'un arbre (qui contrôle directement le biais/variance d'un modèle unique), n_estimators ne fait donc pas sur-apprendre : au pire, on gaspille du temps de calcul pour un gain marginal.

In [ ]:
# Effet de max_features sur la corrélation entre arbres et la performance.
X, y = make_classification(n_samples=600, n_features=20, n_informative=10, n_redundant=5,
                            n_clusters_per_class=2, random_state=RNG)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=RNG, stratify=y)

max_features_list = [1, 2, 5, 10, 20]
mean_corr, test_acc = [], []
for mf in max_features_list:
    forest = RandomForestClassifier(n_estimators=100, max_features=mf, random_state=RNG)
    forest.fit(X_train, y_train)
    test_acc.append(accuracy_score(y_test, forest.predict(X_test)))

    tree_probas = np.array([t.predict_proba(X_test)[:, 1] for t in forest.estimators_])
    corr = np.corrcoef(tree_probas)
    n_t = corr.shape[0]
    mean_corr.append((corr.sum() - n_t) / (n_t * (n_t - 1)))

fig, ax1 = plt.subplots(figsize=(7, 5))
ax1.plot(max_features_list, mean_corr, marker="o", color="tab:blue")
ax1.set_xlabel("max_features (sur 20 variables au total)")
ax1.set_ylabel("Corrélation moyenne entre arbres", color="tab:blue")
ax1.tick_params(axis="y", labelcolor="tab:blue")

ax2 = ax1.twinx()
ax2.plot(max_features_list, test_acc, marker="s", color="tab:red")
ax2.set_ylabel("Accuracy sur le test", color="tab:red")
ax2.tick_params(axis="y", labelcolor="tab:red")

plt.title("Effet de max_features sur la corrélation entre arbres\net sur la performance")
plt.tight_layout()
plt.show()


$$ $$

**Question 4 :** Comment évolue la corrélation moyenne entre arbres lorsque max_features augmente ? Cela va-t-il dans le sens attendu pour la réduction de variance ? L'accuracy sur le test s'améliore-t-elle nécessairement quand max_features augmente ?

$$ $$

*Éléments de réponse.* La corrélation moyenne entre les prédictions des arbres augmente avec max_features : avec plus de variables disponibles à chaque split, les arbres ont davantage tendance à choisir les mêmes coupures et à se ressembler. Or la réduction de variance du bagging est d'autant plus forte que les modèles combinés sont peu corrélés entre eux ; un max_features trop grand (proche de max_features=None, c'est-à-dire toutes les variables) réduit donc la diversité de la forêt. Cela ne se traduit pas forcément par une meilleure accuracy sur le test : au-delà d'un certain point, la performance plafonne, voire se dégrade légèrement, un bon compromis se situant souvent autour de $\sqrt d$.

In [ ]:
# Importance des variables : arbre unique vs forêt aléatoire.
data = load_breast_cancer()
X_bc, y_bc = data.data, data.target
feature_names = data.feature_names

tree = DecisionTreeClassifier(random_state=RNG)
tree.fit(X_bc, y_bc)

forest = RandomForestClassifier(n_estimators=300, random_state=RNG)
forest.fit(X_bc, y_bc)

order = np.argsort(forest.feature_importances_)[::-1][:10]  # 10 variables les plus importantes (RF), sur 30 au total

fig, ax = plt.subplots(figsize=(9, 6))
y_pos = np.arange(len(order))
width = 0.4
ax.barh(y_pos - width / 2, forest.feature_importances_[order], height=width, label="Forêt aléatoire (300 arbres)")
ax.barh(y_pos + width / 2, tree.feature_importances_[order], height=width, label="Arbre unique")
ax.set_yticks(y_pos)
ax.set_yticklabels(np.array(feature_names)[order])
ax.invert_yaxis()
ax.set_xlabel("Importance de la variable")
ax.set_title("Importance des variables (top 10) sur Breast Cancer :\narbre unique vs forêt aléatoire")
ax.legend()
plt.tight_layout()
plt.show()


$$ $$

**Question 5 :** L'ordre et les valeurs d'importance sont-ils identiques entre l'arbre unique et la forêt ? Laquelle des deux estimations vous semble la plus fiable, et pourquoi ?

$$ $$

*Éléments de réponse.* Les deux classements se ressemblent globalement, mais l'arbre unique accorde parfois un poids très marqué (voire quasi exclusif) à une seule variable, alors que la forêt répartit l'importance de façon plus lissée sur plusieurs variables corrélées entre elles. C'est attendu : un arbre unique fait un choix arbitraire (souvent parmi plusieurs variables informatives équivalentes) dès le premier split, alors que la forêt, en moyennant sur de nombreux arbres appris sur des sous-échantillons différents de variables et d'exemples, donne une estimation plus stable et donc plus fiable de l'importance réelle de chaque variable.

### Exercice 4 — Importance des variables sur Wine

Sur `load_wine`, entraînez un `RandomForestClassifier(n_estimators=300, random_state=42)` sur l'ensemble des données, et affichez le nom et la valeur des 3 variables les plus importantes, triées par importance décroissante.


In [ ]:
data = load_wine()
X_wine, y_wine = data.data, data.target

forest = RandomForestClassifier(n_estimators=300, random_state=42)
forest.fit(X_wine, y_wine)

order = np.argsort(forest.feature_importances_)[::-1][:3]
for rank, idx in enumerate(order, start=1):
    print(f"{rank}. {data.feature_names[idx]:30s} importance = {forest.feature_importances_[idx]:.3f}")


### Exercice 5 (niveau Master) — Réglage de `n_estimators` et `max_features`

Sur `load_digits` : séparez les données en train/test (70 % / 30 %, `random_state=42`, `stratify=y`), définissez une grille pour `n_estimators` (par exemple `[50, 150]`) et `max_features` (par exemple `["sqrt", "log2"]`), et utilisez `GridSearchCV` (`cv=5`) pour trouver la meilleure combinaison. Affichez `best_params_` ainsi que l'accuracy du meilleur modèle sur le test.


In [ ]:
X_digits, y_digits = load_digits(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X_digits, y_digits, test_size=0.3, random_state=42, stratify=y_digits)

param_grid = {
    "n_estimators": [50, 150],
    "max_features": ["sqrt", "log2"],
}

grid = GridSearchCV(RandomForestClassifier(random_state=42), param_grid, cv=5)
grid.fit(X_train, y_train)

test_acc = accuracy_score(y_test, grid.best_estimator_.predict(X_test))

print("Meilleurs hyperparamètres :", grid.best_params_)
print(f"Accuracy sur le test      : {test_acc:.3f}")


### Exercice 6 (niveau Master) — Comparaison par validation croisée

Sur `load_breast_cancer`, comparez par validation croisée à 10 plis (`cross_val_score`, `scoring="accuracy"`) un `DecisionTreeClassifier` non contraint et un `RandomForestClassifier(n_estimators=200, random_state=42)`. Affichez la moyenne et l'écart-type des scores pour chacun des deux modèles.


In [ ]:
X_bc, y_bc = load_breast_cancer(return_X_y=True)

scores_tree = cross_val_score(
    DecisionTreeClassifier(random_state=42), X_bc, y_bc, cv=10, scoring="accuracy")
scores_forest = cross_val_score(
    RandomForestClassifier(n_estimators=200, random_state=42), X_bc, y_bc, cv=10, scoring="accuracy")

print(f"Arbre unique     : {scores_tree.mean():.3f} (+/- {scores_tree.std():.3f})")
print(f"Forêt aléatoire  : {scores_forest.mean():.3f} (+/- {scores_forest.std():.3f})")

# La forêt aléatoire obtient en général une accuracy moyenne plus élevée et
# un écart-type plus faible entre les plis : elle est à la fois plus
# performante et plus stable que l'arbre unique.


### Exercice 7 (niveau Master) — Forêt aléatoire en régression

Les forêts aléatoires s'appliquent aussi à la régression (`RandomForestRegressor`), en remplaçant le vote majoritaire par une moyenne. Sur `load_diabetes`, comparez par validation croisée à 5 plis (`cross_val_score`, `scoring="neg_mean_squared_error"`) un `DecisionTreeRegressor` non contraint et un `RandomForestRegressor(n_estimators=200, random_state=42)`. Affichez le RMSE moyen de chacun (racine carrée de l'opposé des scores).


In [ ]:
X_diab, y_diab = load_diabetes(return_X_y=True)

scores_tree = cross_val_score(
    DecisionTreeRegressor(random_state=42), X_diab, y_diab, cv=5, scoring="neg_mean_squared_error")
scores_forest = cross_val_score(
    RandomForestRegressor(n_estimators=200, random_state=42), X_diab, y_diab, cv=5, scoring="neg_mean_squared_error")

rmse_tree = np.sqrt(-scores_tree.mean())
rmse_forest = np.sqrt(-scores_forest.mean())

print(f"RMSE moyen - arbre de régression unique : {rmse_tree:.1f}")
print(f"RMSE moyen - forêt aléatoire             : {rmse_forest:.1f}")

# Comme en classification, la forêt aléatoire réduit la variance de l'arbre
# de régression unique et obtient en général un RMSE moyen plus faible.
assert rmse_tree > 0 and rmse_forest > 0


### Exercice 8 (niveau Master) — Le bagging profite-t-il à tous les modèles de base ?

Le bagging réduit d'autant plus l'erreur que les modèles de base ont une **variance élevée**. Un k-NN a en général une variance plus faible qu'un arbre profond (surtout pour un nombre de voisins raisonnable) : le bagging devrait donc lui apporter un gain plus faible. Sur un jeu de données `make_classification(n_samples=400, n_features=20, n_informative=15, random_state=42)`, comparez par validation croisée à 5 plis un `BaggingClassifier(estimator=DecisionTreeClassifier(), n_estimators=50, random_state=42)` à un `BaggingClassifier(estimator=KNeighborsClassifier(n_neighbors=5), n_estimators=50, random_state=42)`, ainsi qu'aux deux modèles de base pris isolément (sans bagging). Le gain apporté par le bagging est-il comparable pour les deux types de modèles de base ?


In [ ]:
X, y = make_classification(n_samples=400, n_features=20, n_informative=15, random_state=42)

models = {
    "Arbre seul": DecisionTreeClassifier(random_state=42),
    "Bagging d'arbres": BaggingClassifier(estimator=DecisionTreeClassifier(), n_estimators=50, random_state=42),
    "k-NN seul": KNeighborsClassifier(n_neighbors=5),
    "Bagging de k-NN": BaggingClassifier(estimator=KNeighborsClassifier(n_neighbors=5), n_estimators=50, random_state=42),
}

results = {name: cross_val_score(model, X, y, cv=5).mean() for name, model in models.items()}

for name, score in results.items():
    print(f"{name:20s} : accuracy moyenne = {score:.3f}")

gain_arbre = results["Bagging d'arbres"] - results["Arbre seul"]
gain_knn = results["Bagging de k-NN"] - results["k-NN seul"]
print(f"\nGain du bagging pour l'arbre : {gain_arbre:+.3f}")
print(f"Gain du bagging pour le k-NN : {gain_knn:+.3f}")

# Le bagging apporte en général un gain plus net pour l'arbre de décision
# (modèle instable, forte variance) que pour le k-NN (déjà plus stable en
# lui-même), conformément à l'intuition théorique : le bagging profite
# surtout aux modèles de base à forte variance.


## Pour la suite

Le bagging et les forêts aléatoires réduisent la variance en combinant, en parallèle, des modèles instables entraînés indépendamment. Le notebook suivant (9/9) aborde une logique très différente : le **boosting**, qui combine séquentiellement des apprenants faibles pour réduire le biais.
